# 🔍 Face Scanner Attendance System

ระบบแสกนใบหน้าสำหรับลงเวลาเข้างาน — ทำงานบน Google Colab พร้อม Google Drive สำหรับเก็บข้อมูล

## ⚠️ ข้อจำกัดของ Colab
- **Session หมดอายุ** ประมาณ 12 ชั่วโมง (Colab Free) / 24 ชั่วโมง (Colab Pro+)
- **URL ngrok เปลี่ยน** ทุกครั้งที่รัน notebook ใหม่ — แต่ไม่กระทบ React เพราะใช้ relative URL
- **ข้อมูลบันทึกลง Google Drive** ไม่หายเมื่อ session รีสตาร์ท
- ต้องรัน notebook ใหม่ทุกวัน (แนะนำ Colab Pro+ เพื่อ session ที่ยาวขึ้น)

## 🔐 Security Notes
- ระบบใช้ JWT สำหรับ authenticate admin และ employee
- Scan page (kiosk) เปิดสาธารณะ — ใช้ใบหน้าเป็น authentication
- เปลี่ยน `JWT_SECRET` ในขั้นตอนที่ 2 ก่อนใช้งานจริง

---
**ลำดับขั้นตอน:** Cell 1 → Cell 2 → Cell 3 → Cell 4 → Cell 5 → เปิด URL ที่แสดงใน Cell 5

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 1: ติดตั้ง dependencies
# รันครั้งแรกเท่านั้น (หลังจากนั้น Colab cache ไว้แล้ว)
# ═══════════════════════════════════════════════════════

# ตรวจสอบว่ามี GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU พร้อมใช้งาน')
    print(result.stdout.split('\n')[8])
else:
    print('⚠️  ไม่พบ GPU — ระบบจะทำงานช้าลง เปลี่ยนเป็น Runtime > Change runtime type > T4 GPU')

# ติดตั้ง packages — ใช้ minimum versions เพื่อหลีกเลี่ยง conflict กับ Colab pre-installed packages
!pip install -q \
    "fastapi>=0.115.2,<1.0" \
    "uvicorn[standard]>=0.30.0,<1.0" \
    "python-multipart>=0.0.18,<1.0" \
    "insightface>=0.7.3" \
    "onnxruntime-gpu>=1.19.0" \
    "numpy>=2.0" \
    "Pillow>=10.0" \
    "pyngrok>=7.0" \
    "python-dotenv>=1.0" \
    "aiofiles>=23.0" \
    "python-jose[cryptography]>=3.3.0,<4.0" \
    "passlib[bcrypt]>=1.7.4,<2.0"

print('\n✅ ติดตั้ง dependencies เสร็จแล้ว')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 2: Mount Google Drive + ตั้งค่า environment
# ═══════════════════════════════════════════════════════

import os
from google.colab import drive

drive.mount('/content/drive')

# สร้างโฟลเดอร์บน Drive
DRIVE_DIR = '/content/drive/MyDrive/face_scanner'
PHOTOS_DIR = f'{DRIVE_DIR}/photos'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(PHOTOS_DIR, exist_ok=True)

print(f'✅ Google Drive mounted')
print(f'📁 ข้อมูลจะถูกเก็บที่: {DRIVE_DIR}')
print(f'🖼️  รูปภาพจะถูกเก็บที่: {PHOTOS_DIR}')

# ═══════════════════════════════════════════════════════
# ⚠️  เปลี่ยน JWT_SECRET ก่อนใช้งานจริง!
# สร้าง secret แบบ random: import secrets; print(secrets.token_hex(32))
# ═══════════════════════════════════════════════════════
os.environ['DB_PATH'] = f'{DRIVE_DIR}/attendance.db'
os.environ['PHOTOS_DIR'] = PHOTOS_DIR
os.environ['JWT_SECRET'] = 'CHANGE-THIS-TO-A-RANDOM-64-CHAR-HEX-STRING-BEFORE-PRODUCTION'
os.environ['PORT'] = '8000'

print('\n✅ Environment variables ตั้งค่าเสร็จแล้ว')
print('⚠️  อย่าลืมเปลี่ยน JWT_SECRET ก่อนใช้งานจริง!')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 3: โหลด code จาก GitHub
# ═══════════════════════════════════════════════════════

import os

REPO_URL = 'https://github.com/YOUR_USERNAME/face-scanner.git'  # เปลี่ยนเป็น URL repo ของคุณ
APP_DIR = '/content/face_scanner'

if os.path.exists(APP_DIR):
    print('📁 พบโฟลเดอร์ app แล้ว — pull โค้ดล่าสุด')
    !cd {APP_DIR} && git pull
else:
    print('📥 Clone repository...')
    !git clone {REPO_URL} {APP_DIR}

print(f'\n✅ โค้ดพร้อมใช้งานที่ {APP_DIR}')

# Build React frontend
print('\n🔨 Build React frontend...')
!cd {APP_DIR}/frontend && npm install --silent && npm run build
print('✅ React frontend built เสร็จแล้ว')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 4: รัน FastAPI server + ngrok tunnel
# รัน cell นี้ทุกครั้งที่เปิด session ใหม่
# ═══════════════════════════════════════════════════════

import threading
import time
import sys

APP_DIR = '/content/face_scanner'
sys.path.insert(0, f'{APP_DIR}/backend')

# รัน FastAPI ใน background thread
def run_server():
    import uvicorn
    uvicorn.run(
        'main:app',
        host='0.0.0.0',
        port=8000,
        log_level='warning',
    )

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# รอ server เริ่มทำงาน
print('⏳ รอ server เริ่มทำงาน...')
time.sleep(5)

# ตรวจสอบ health
import urllib.request
import json

try:
    response = urllib.request.urlopen('http://localhost:8000/health', timeout=10)
    data = json.loads(response.read())
    print(f'✅ Server ทำงานแล้ว — {data}')
except Exception as e:
    print(f'❌ Server ยังไม่พร้อม: {e}')
    print('ลอง restart runtime แล้วรันใหม่')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 5: เปิด ngrok tunnel + แสดง URL
# ═══════════════════════════════════════════════════════

from pyngrok import ngrok, conf
import getpass

# ─── ตั้งค่า ngrok authtoken ─────────────────────────────
# สมัคร free account ที่ https://ngrok.com แล้วเอา authtoken มาใส่
# หรือรันด้านล่างเพื่อ input แบบ secure:
# NGROK_TOKEN = getpass.getpass('กรอก ngrok authtoken: ')
NGROK_TOKEN = 'YOUR_NGROK_AUTHTOKEN'  # แทนที่ด้วย token จริง

conf.get_default().auth_token = NGROK_TOKEN

# เปิด tunnel
public_url = ngrok.connect(8000)
app_url = str(public_url)

print('=' * 60)
print('🌐 ระบบพร้อมใช้งาน!')
print('=' * 60)
print(f'🔗 URL: {app_url}')
print(f'📱 สแกน QR หรือเปิดลิงก์บนอุปกรณ์ใดก็ได้ในเครือข่าย')
print('=' * 60)
print()
print('📋 ขั้นตอนแรก:')
print(f'   1. เปิด {app_url}/setup')
print('   2. สร้าง admin account (ทำแค่ครั้งแรก)')
print('   3. Login → Admin Panel → เพิ่มพนักงาน')
print()
print('🖥️  Kiosk (สแกนหน้า — ไม่ต้อง login):')
print(f'   {app_url}/scan')
print()
print('⚠️  URL นี้จะเปลี่ยนทุกครั้งที่รัน session ใหม่')
print('⚠️  Session จะหมดอายุใน ~12 ชม. (Colab Free) / ~24 ชม. (Colab Pro+)')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 6: Keep-alive (ป้องกัน Colab หมด session เร็วเกินไป)
# รัน cell นี้ค้างไว้เพื่อ prevent idle timeout
# กด Stop (■) เมื่อต้องการหยุด
# ═══════════════════════════════════════════════════════

import time
import urllib.request
import json
from datetime import datetime

print('🔄 Keep-alive เริ่มทำงาน — กด ■ เพื่อหยุด')
print()

ping_interval_seconds = 300  # ping ทุก 5 นาที
ping_count = 0

while True:
    try:
        response = urllib.request.urlopen('http://localhost:8000/health', timeout=5)
        data = json.loads(response.read())
        ping_count += 1
        now = datetime.now().strftime('%H:%M:%S')
        if ping_count % 12 == 0:  # แสดงทุกชั่วโมง
            print(f'[{now}] ✅ Server ทำงานปกติ — ping #{ping_count}')
    except Exception as e:
        print(f'⚠️  Server ไม่ตอบสนอง: {e}')

    time.sleep(ping_interval_seconds)

## 🛠️ Admin Tools

รัน cells ด้านล่างเมื่อต้องการจัดการระบบโดยตรงจาก Colab

In [ ]:
# แสดงรายชื่อผู้ใช้ทั้งหมดในระบบ
import sqlite3, os

db_path = os.environ.get('DB_PATH', '/content/drive/MyDrive/face_scanner/attendance.db')
conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row

print('👥 รายชื่อผู้ใช้:')
for row in conn.execute('SELECT u.name, u.employee_id, u.created_at, COUNT(e.id) as face_count FROM users u LEFT JOIN face_embeddings e ON e.user_id = u.id GROUP BY u.id').fetchall():
    print(f'  • {row["name"]} (ID: {row["employee_id"] or "-"}) — {row["face_count"]} face samples — สมัครเมื่อ {row["created_at"][:10]}')

print()
print('📊 Attendance วันนี้:')
from datetime import date
today = date.today().isoformat()
rows = conn.execute(
    'SELECT u.name, a.timestamp, a.confidence FROM attendance a JOIN users u ON u.id = a.user_id WHERE a.timestamp LIKE ? ORDER BY a.timestamp',
    (f'{today}%',)
).fetchall()
if rows:
    for row in rows:
        print(f'  • {row["name"]} เข้างาน {row["timestamp"][11:16]} (confidence: {row["confidence"]:.1%})')
else:
    print('  ยังไม่มีการสแกนวันนี้')

conn.close()

In [ ]:
# Export attendance เป็น CSV
import sqlite3, os
import pandas as pd
from google.colab import files

db_path = os.environ.get('DB_PATH', '/content/drive/MyDrive/face_scanner/attendance.db')
conn = sqlite3.connect(db_path)

df = pd.read_sql_query(
    '''
    SELECT u.name as ชื่อ, u.employee_id as รหัสพนักงาน,
           SUBSTR(a.timestamp, 1, 10) as วันที่,
           SUBSTR(a.timestamp, 12, 5) as เวลาเข้างาน,
           ROUND(a.confidence * 100, 1) as ความมั่นใจ_เปอร์เซ็นต์
    FROM attendance a
    JOIN users u ON u.id = a.user_id
    ORDER BY a.timestamp DESC
    ''',
    conn
)

export_path = '/content/attendance_export.csv'
df.to_csv(export_path, index=False, encoding='utf-8-sig')
conn.close()

print(f'✅ Export เสร็จแล้ว — {len(df)} รายการ')
files.download(export_path)